# Data Ingestion & Validation
Fetch, inspect, and validate raw data before running the balance model.

**Run this notebook first** to download the STEO workbook and verify data quality.

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np
from pathlib import Path
from dotenv import load_dotenv
import os

load_dotenv("../.env")
EIA_API_KEY = os.getenv("EIA_API_KEY", "")

from src.eia_client import EIAClient
from src.steo_loader import STEOLoader
from src.opec_loader import OPECLoader
from src.iea_loader import IEALoader

print(f"EIA API Key configured: {'Yes' if EIA_API_KEY and EIA_API_KEY != 'your_api_key_here' else 'No — register at https://www.eia.gov/opendata/register.php'}")

## 1. Download & Parse EIA STEO
The STEO workbook is the backbone of the model — a single Excel file with the complete global oil balance.

In [ ]:
steo = STEOLoader(cache_dir="../data/raw/steo")

# Download latest STEO workbook (auto-skips if cached copy is < 7 days old)
steo_path = steo.download()

# Parse into a clean DataFrame (also extracts metadata from Dates sheet)
steo_data = steo.parse(steo_path, start="2023-01", end="2026-12")

# Show auto-detected metadata
meta = steo.metadata
print(f"STEO Edition:          {meta['forecast_month']}")
print(f"Modeling Date:         {meta['modeling_date'].strftime('%Y-%m-%d') if meta['modeling_date'] else 'N/A'}")
print(f"Last Historical Month: {meta['last_historical_month_str']}  ← auto-detected")
print(f"\nData range: {steo_data.index.min().strftime('%Y-%m')} to {steo_data.index.max().strftime('%Y-%m')}")
print(f"Columns: {steo_data.columns.tolist()}")
print(f"Shape: {steo_data.shape}")
steo_data.head(12)

In [ ]:
# Quick sanity check — numbers should be in expected ranges
print("=== STEO Data Sanity Check ===")
for col in steo_data.columns:
    vals = steo_data[col].dropna()
    if not vals.empty:
        print(f"{col:30s}  min={vals.min():8.2f}  max={vals.max():8.2f}  mean={vals.mean():8.2f}")

## 2. EIA API — US Crude Inventories
Weekly inventory data resampled to monthly. Requires an API key.

In [ ]:
if EIA_API_KEY and EIA_API_KEY != "your_api_key_here":
    eia = EIAClient(api_key=EIA_API_KEY)
    us_stocks = eia.get_monthly_crude_stocks(start="2023-01")
    print(f"US crude stocks: {len(us_stocks)} months")
    print(f"Range: {us_stocks.index.min()} to {us_stocks.index.max()}")
    print(f"\nLatest 6 months:")
    display(us_stocks.tail(6))
else:
    print("Skipping EIA API — no API key configured.")
    print("Register at: https://www.eia.gov/opendata/register.php")
    print("Then update .env file with your key.")
    us_stocks = pd.Series(dtype=float, name="us_crude_stocks_mmbbl")

## 3. OPEC MOMR Data
Check if OPEC production data has been entered in the CSV template.

In [ ]:
opec = OPECLoader()

opec_prod_path = "../data/templates/opec_production_template.csv"
if opec.has_data(opec_prod_path):
    opec_prod = opec.load_production_csv(opec_prod_path)
    print(f"OPEC production data: {len(opec_prod)} months with data")
    display(opec_prod.dropna(how='all').tail())
else:
    print("No OPEC production data entered yet.")
    print(f"Fill in: {opec_prod_path}")
    print("Data source: OPEC MOMR Table 5.13 (secondary sources)")
    print("Download from: https://www.opec.org/monthly-oil-market-report.html")

## 4. IEA OMR Data
Check if IEA data has been entered in the CSV templates.

In [ ]:
iea = IEALoader()

iea_supply_path = "../data/templates/iea_supply_template.csv"
iea_demand_path = "../data/templates/iea_demand_template.csv"

iea_supply = iea.load_supply_csv(iea_supply_path)
iea_demand = iea.load_demand_csv(iea_demand_path)

has_iea_supply = iea.has_data(iea_supply)
has_iea_demand = iea.has_data(iea_demand)

print(f"IEA supply data available: {has_iea_supply}")
print(f"IEA demand data available: {has_iea_demand}")

if not has_iea_supply and not has_iea_demand:
    print("\nNo IEA data entered. The model will use STEO as the sole source.")
    print("To add IEA data, fill in the templates:")
    print(f"  Supply: {iea_supply_path}")
    print(f"  Demand: {iea_demand_path}")

## 5. Save Processed Data
Cache the cleaned STEO data and inventories for the main model notebook.

In [ ]:
import json

processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

steo_data.to_csv(processed_dir / "steo_data.csv")
print(f"Saved STEO data: {processed_dir / 'steo_data.csv'}")

if not us_stocks.empty:
    us_stocks.to_csv(processed_dir / "us_crude_stocks.csv")
    print(f"Saved US stocks: {processed_dir / 'us_crude_stocks.csv'}")

# Save metadata so the balance model notebook can read it without re-parsing the workbook
if steo.metadata:
    meta_save = {
        "forecast_month": steo.metadata["forecast_month"],
        "modeling_date": steo.metadata["modeling_date"].isoformat() if steo.metadata["modeling_date"] else None,
        "last_historical_month": steo.metadata["last_historical_month_str"],
    }
    with open(processed_dir / "steo_metadata.json", "w") as f:
        json.dump(meta_save, f, indent=2)
    print(f"Saved STEO metadata: {processed_dir / 'steo_metadata.json'}")

print("\nDone! Proceed to 02_oil_balance_model.ipynb")